In [1]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tabulate import tabulate
import collections
import pickle,os,csv
from pathlib import Path  
import itertools
import seaborn as sns
import glob
import os
from scipy import stats

In [2]:
# import models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# import model evaluation metrics
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import imblearn
from imblearn.under_sampling import RandomUnderSampler
from sklearn.utils import resample
from sklearn.metrics import accuracy_score


In [3]:
# import datasets

synonoms = 'drugbank_vocabulary.csv'
drug_syn = pd.read_csv(synonoms)
drug_sider = pd.read_csv('drug_names.tsv', sep='\t')
drug_SE = pd.read_csv('meddra_all_se.tsv', sep='\t')
DB_summary = pd.read_csv('drugbank.tsv', sep='\t')

In [4]:
# Generate drug names to DBID dictionary 
# ensures that both common names and synonyms of a drug can be used to look up its DrugBank ID

drug_syn['synlist'] = ""
for i in range(0, len(drug_syn['Synonyms'])):
    drug_syn.loc[i, 'synlist'] = ", ".join(str(drug_syn.loc[i, 'Synonyms']).lower().split(" | "))
    drug_syn.loc[i, 'Common name'] = drug_syn.loc[i, 'Common name'].lower()

from collections import defaultdict
chemical_to_DBID=defaultdict(set)


for (drug, DBID) in zip(drug_syn['Common name'], drug_syn['DrugBank ID']):
  
  chemical_to_DBID[drug].add(DBID)

for(drug, DBID) in zip(drug_syn['synlist'], drug_syn['DrugBank ID']):
  for i in range(0, len(drug)):
    chemical_to_DBID[drug[i]].add(DBID)

In [5]:
##SIDER drug name dataset processing##
# Standardizes drug names and links SIDER drugs to DrugBank for integration

#Rearrange rows and rename columns

drug_sider.loc[-1] = drug_sider.columns.values
drug_sider.sort_index(inplace=True)
drug_sider.reset_index(drop=True, inplace=True)
drug_sider.columns=['drugID', 'drugname']

#Map SIDER drugnames to DBID

for i in range(len(drug_sider['drugname'])):
    drug_sider.loc[i, 'drugname']=drug_sider.loc[i, 'drugname'].lower()

drug_sider['DBID'] = ''
drug_sider['DBID'] = drug_sider['drugname'].map(chemical_to_DBID)

for i in range(len(drug_sider['DBID'])):
    drug_sider.loc[i, 'DBID'] = str(drug_sider.loc[i, 'DBID']).replace('{',"").replace('}',"").replace("'","")
    
#Determine number of DBID match

match_bool=drug_sider['DBID'] !='set()'
match_index = match_bool[match_bool].index

print('The number of matched DBIDS is', sum(drug_sider['DBID'] !='set()'))
print('The number of unmatched DBIDS is', sum(drug_sider['DBID'] =='set()'))

print(drug_sider.head())

The number of matched DBIDS is 1095
The number of unmatched DBIDS is 335
         drugID                  drugname     DBID
0  CID100000085                 carnitine    set()
1  CID100000119        gamma-aminobutyric    set()
2  CID100000137          5-aminolevulinic    set()
3  CID100000143                leucovorin  DB00650
4  CID100000146  5-methyltetrahydrofolate    set()


In [6]:
# Generate SIDER DrugID to DBID - this will map drug_SE drugID to its respective DBID
drugID_to_DBID={}

for i in range(len(drug_sider['drugID'])):
    drugID_to_DBID[drug_sider.loc[i, 'drugID']] = drug_sider.loc[i, 'DBID']

In [7]:
# run to get drug_SE data frame
drug_SE = pd.read_excel('./intermediate_data/drug_side_effects.xlsx')

In [8]:
# Generate the list of top 30 side effects in SIDER 4.1
from collections import Counter

side_effect_count = pd.DataFrame(Counter(drug_SE['side_effect']).most_common(30), columns=['Side Effect', 'Drug Count',])

In [9]:
#Generate Dictionary of top 30 side effects with its associated DrugBank ID

from collections import defaultdict

sid_to_dbid = defaultdict(list)
for j in side_effect_count['Side Effect']:
    for i in range(len(drug_SE['side_effect'])):
        if drug_SE['side_effect'][i] == j:
            sid_to_dbid['phen_ind_'+str(j)].append(drug_SE['DBID'][i])

for key, item in sid_to_dbid.items():
    sid_to_dbid[key] = list(set([x for x in item if str(x) !='set()']))

# Functions

In [10]:
#Matrix Generation

def matrix(dic):
    all_rows=[]
    for (drug, dt_set) in dic.items():
        row_data = {'DrugName': drug}
        for dt in dt_set:
            row_data[dt] = 1
        all_rows.append(row_data)
    dt_df = pd.DataFrame(all_rows)
    dt_df = dt_df.fillna(0)
    dt_df = dt_df[dt_df['DrugName'].str.contains('DB')]
    return dt_df

In [11]:
# feature matrix for approved drugs only
def matrix_approved(dic):
    approved_index = []
    all_rows=[]
    for (drug, dt_set) in dic.items():
        row_data = {'DrugName': drug}
        for dt in dt_set:
            row_data[dt] = 1
        all_rows.append(row_data)
    dt_df = pd.DataFrame(all_rows)
    dt_df = dt_df.fillna(0)
    dt_df = dt_df[dt_df['DrugName'].str.contains('DB')]
    for i, k in enumerate(dt_df['DrugName']):
        for j in approved_drugs:
            if k == j:
                approved_index.append(i)
    dt_df_approved = dt_df.iloc[approved_index]
    return dt_df_approved

In [12]:
#Model Inputs
# Transforms the dataset into a machine-learning-ready format, 
# where X is the feature set, and sid_eff_pred contains labels for predicting side effects
def model_input(matrix):
    for key, value in sid_to_dbid.items():
        matrix[key]=''
    for key, value in sid_to_dbid.items():
        for i in value:
            matrix.loc[matrix['DrugName'] != i, key] = 0
    for key, value in sid_to_dbid.items():
        for i in value:
            matrix.loc[matrix['DrugName'] == i, key] = 1
    key_all = []
    for key, value in sid_to_dbid.items():
        key_all.append(key)
        
    X = matrix.drop(key_all, axis=1)
    X = X.drop(['DrugName'], axis=1).values
    
    sid_eff_pred = {}
    for j in side_effect_count['Side Effect']:
        sid_eff_pred[j] = matrix['phen_ind_'+str(j)].astype('int')
    
    return X, sid_eff_pred

In [13]:
#Logistic Regression Bootstrapped 100 Times

# Runs the model multiple times 
# splits 80% training, 20% testing
# trains a model for each side effect
# predicts and stores accuracy scores in boot_df

def log_reg_boot100(X, Y):
    bootstrapNum = 100
    boot_df = pd.DataFrame()
    for k , j in Y.items():
        for i in range (bootstrapNum):
            log_reg = LogisticRegression()
            rus = RandomUnderSampler() # balance classes so model learns to recognize features that actually lead to the side effect, instead of just defaulting to "0" (majority class)
            X_sample, Y_sample = rus.fit_resample(X, j)
            X_train, X_test, Y_train, Y_test = train_test_split(X_sample, Y_sample, train_size = 0.8, test_size=0.2, random_state=1)
            log_reg.fit(X_train, Y_train)
            Y_pred = log_reg.predict(X_test)
            boot_df = pd.concat((boot_df, pd.DataFrame({k: accuracy_score(Y_test, Y_pred)}, index=[i])))
    boot_df = boot_df.apply(lambda x: pd.Series(x.dropna().values))
    return boot_df

In [14]:
# Isolate approved drugs and their targets 
a2n = pickle.load(open('Pfx050120_dint.pkl', 'rb'))

# modified below (caused a key error before)
for i in range(len(DB_summary['groups'])):
    DB_summary['groups'][i] = DB_summary['groups'][i].split('|')

approved_drugs = set()
approved_dbids = {DB_summary['drugbank_id'][i] for i, group in enumerate(DB_summary['groups']) for drug in group if drug == 'approved'}
for dbid, targets in a2n.items():
    if dbid in approved_dbids:
        approved_drugs.add(dbid)

/var/folders/0x/qp6y3wx113b2t9y010wfvr5h0000gn/T/ipykernel_93295/429280400.py:6: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  DB_summary['groups'][i] = DB_summary['groups'][i].split('|')


In [15]:
# creates separate datasets for all drugs vs. approved drugs to compare side effect predictions
targets_approved = matrix_approved(a2n)
targets = matrix(a2n)

X, Y = model_input(targets)
X_approved, Y_approved = model_input(targets_approved)

targets_approved.head()

,DrugName,MAPK10,astB,CYP2B6,TTR,CYP3A4,ABCG2,SLC6A3,SLC6A4,CYP2D6,...,phen_ind_insomnia,phen_ind_anaphylactic shock,phen_ind_paraesthesia,phen_ind_somnolence,phen_ind_nervous system disorder,phen_ind_thrombocytopenia,phen_ind_tachycardia,phen_ind_arthralgia,phen_ind_infection,phen_ind_musculoskeletal discomfort
4,DB00285,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,...,1,1,1,1,0,1,1,1,1,1
7,DB00648,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0,0,1,1,1,1,0,0,1,0
16,DB00043,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
17,DB00417,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
22,DB09097,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
# shows how many drugs were mapped to a side effect

side_effect_count['DBID Match Count'] = ''
# side_effect_count['Matrix Match Count'] = ''
for i, j in enumerate(side_effect_count['Side Effect']):
    side_effect_count.loc[i, 'DBID Match Count'] = len(sid_to_dbid['phen_ind_'+str(j)])

for i, j in enumerate(side_effect_count['Side Effect']):
    side_effect_count.loc[i, 'Matrix Match Count'] = targets_approved['phen_ind_'+str(j)].value_counts()[1]

# ATC Levels Only Comparison

In [18]:
# ATC Codes ONLY
# load in Level 2

ATC_only = pd.read_excel("./intermediate_data/atc_only.xlsx")

   Unnamed: 0 DrugName  N06  L01  R03  J01  G02  J02  D01  G03  ...  A16  A09  \
0           3  DB00285    1    0    0    0    0    0    0    0  ...    0    0   
1           4  DB00648    0    1    0    0    0    0    0    0  ...    0    0   
2           7  DB00043    0    0    1    0    0    0    0    0  ...    0    0   
3           8  DB00417    0    0    0    1    0    0    0    0  ...    0    0   
4           9  DB09097    0    0    0    0    1    0    0    0  ...    0    0   

   R07  V10  J04  J06  D02  D09  H04  V06  
0    0    0    0    0    0    0    0    0  
1    0    0    0    0    0    0    0    0  
2    0    0    0    0    0    0    0    0  
3    0    0    0    0    0    0    0    0  
4    0    0    0    0    0    0    0    0  

[5 rows x 90 columns]


In [19]:
# Model Evaluation with ATC Level 2 Codes Only

X, Y = model_input(ATC_only)
LR_ATC = log_reg_boot100(X, Y)
for k , j in Y.items():
    print('This is the LR mean accuracy on x100 bootstrap for '+str(k), LR_ATC[k].mean())

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

This is the LR mean accuracy on x100 bootstrap for dizziness 0.6794915254237288
This is the LR mean accuracy on x100 bootstrap for nausea 0.6843712574850299
This is the LR mean accuracy on x100 bootstrap for headache 0.6646518987341774
This is the LR mean accuracy on x100 bootstrap for rash 0.6637417218543046
This is the LR mean accuracy on x100 bootstrap for vomiting 0.6949013157894737
This is the LR mean accuracy on x100 bootstrap for asthenia 0.7026591760299626
This is the LR mean accuracy on x100 bootstrap for diarrhoea 0.6975000000000001
This is the LR mean accuracy on x100 bootstrap for pruritus 0.6991044776119404
This is the LR mean accuracy on x100 bootstrap for hypersensitivity 0.6828400000000001
This is the LR mean accuracy on x100 bootstrap for abdominal pain 0.6976170212765958
This is the LR mean accuracy on x100 bootstrap for urticaria 0.6788596491228072
This is the LR mean accuracy on x100 bootstrap for body temperature increased 0.7145089285714286
This is the LR mean acc

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

In [21]:
# ATC Codes ONLY
# repeat for Level 3
    
# Create DBID to ATC dictionary
DBatc = defaultdict(set)

for dbid, targets in a2n.items(): # changed to a2n because omitting PathFX, hopefully this is correct someone please double check
    for i in range(len(DB_summary['atc_codes'])):
        DB_atc = set()
        if dbid == DB_summary['drugbank_id'][i]:
            for j in range(len(DB_summary['atc_codes'][i])):
                DB_atc.add(DB_summary['atc_codes'][i][j][0:4])
            DBatc[DB_summary['drugbank_id'][i]] = DB_atc

# Generate Matrix
ATC_only_3 = matrix_approved(DBatc)
ATC_only_3 = ATC_only_3[ATC_only_3['DrugName'].str.contains('DB')]
ATC_only_3 = ATC_only_3.drop(columns = ['N/A'])
ATC_only_3

,DrugName,N06A,L01X,R03D,J01C,G02C,J01R,J02A,D01A,G03F,...,C03X,G02B,A11A,P02D,D10B,A07X,M03C,V06D,A10X,H02B
3,DB00285,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,DB00648,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,DB00043,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,DB00417,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,DB09097,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6979,DB00577,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6984,DB00800,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6986,DB14542,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6987,DB01551,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
ATC_only_3.to_excel("atc_only_3.xlsx") # saving intermediate data

In [29]:
# Model Evaluation with ATC Level 3 Codes Only

X, Y = model_input(ATC_only_3)
LR_ATC_3 = log_reg_boot100(X, Y)
for k , j in Y.items():
    print('This is the LR mean accuracy on x100 bootstrap for '+str(k), LR_ATC_3[k].mean())

This is the LR mean accuracy on x100 bootstrap for dizziness 0.6649830508474576
This is the LR mean accuracy on x100 bootstrap for nausea 0.680748502994012
This is the LR mean accuracy on x100 bootstrap for headache 0.6799367088607595
This is the LR mean accuracy on x100 bootstrap for rash 0.675662251655629
This is the LR mean accuracy on x100 bootstrap for vomiting 0.6887171052631578
This is the LR mean accuracy on x100 bootstrap for asthenia 0.7197003745318352
This is the LR mean accuracy on x100 bootstrap for diarrhoea 0.6956159420289856
This is the LR mean accuracy on x100 bootstrap for pruritus 0.7122014925373135
This is the LR mean accuracy on x100 bootstrap for hypersensitivity 0.6934400000000001
This is the LR mean accuracy on x100 bootstrap for abdominal pain 0.700936170212766
This is the LR mean accuracy on x100 bootstrap for urticaria 0.6950000000000002
This is the LR mean accuracy on x100 bootstrap for body temperature increased 0.6854910714285714
This is the LR mean accura

In [27]:
# ATC Codes ONLY
# repeat for Level 4
    
# Create DBID to ATC dictionary
DBatc = defaultdict(set)

for dbid, targets in a2n.items(): # changed to a2n because omitting PathFX, hopefully this is correct someone please double check
    for i in range(len(DB_summary['atc_codes'])):
        DB_atc = set()
        if dbid == DB_summary['drugbank_id'][i]:
            for j in range(len(DB_summary['atc_codes'][i])):
                DB_atc.add(DB_summary['atc_codes'][i][j][0:5])
            DBatc[DB_summary['drugbank_id'][i]] = DB_atc

# Generate Matrix
ATC_only_4 = matrix_approved(DBatc)
ATC_only_4 = ATC_only_4[ATC_only_4['DrugName'].str.contains('DB')]
ATC_only_4 = ATC_only_4.drop(columns = ['N/A'])
ATC_only_4

ATC_only_4.to_excel("atc_only_4.xlsx") # saving intermediate data

In [30]:
# Model Evaluation with ATC Level 4 Codes Only

X, Y = model_input(ATC_only_4)
LR_ATC_4 = log_reg_boot100(X, Y)
for k , j in Y.items():
    print('This is the LR mean accuracy on x100 bootstrap for '+str(k), LR_ATC_4[k].mean())

This is the LR mean accuracy on x100 bootstrap for dizziness 0.6900677966101694
This is the LR mean accuracy on x100 bootstrap for nausea 0.678562874251497
This is the LR mean accuracy on x100 bootstrap for headache 0.6830696202531644
This is the LR mean accuracy on x100 bootstrap for rash 0.6834768211920529
This is the LR mean accuracy on x100 bootstrap for vomiting 0.672796052631579
This is the LR mean accuracy on x100 bootstrap for asthenia 0.7112734082397004
This is the LR mean accuracy on x100 bootstrap for diarrhoea 0.7014130434782611
This is the LR mean accuracy on x100 bootstrap for pruritus 0.7047388059701493
This is the LR mean accuracy on x100 bootstrap for hypersensitivity 0.6630800000000001
This is the LR mean accuracy on x100 bootstrap for abdominal pain 0.6891063829787234
This is the LR mean accuracy on x100 bootstrap for urticaria 0.7031140350877194
This is the LR mean accuracy on x100 bootstrap for body temperature increased 0.7058928571428572
This is the LR mean accur

In [33]:
# ATC Codes ONLY
# repeat for Level 5
    
# Create DBID to ATC dictionary
DBatc = defaultdict(set)

for dbid, targets in a2n.items(): # changed to a2n because omitting PathFX, hopefully this is correct someone please double check
    for i in range(len(DB_summary['atc_codes'])):
        DB_atc = set()
        if dbid == DB_summary['drugbank_id'][i]:
            for j in range(len(DB_summary['atc_codes'][i])):
                DB_atc.add(DB_summary['atc_codes'][i][j][0:7])
            DBatc[DB_summary['drugbank_id'][i]] = DB_atc

# Generate Matrix
ATC_only_5 = matrix_approved(DBatc)
ATC_only_5 = ATC_only_5[ATC_only_5['DrugName'].str.contains('DB')]
ATC_only_5 = ATC_only_5.drop(columns = ['N/A'])
ATC_only_5

ATC_only_5.to_excel("atc_only_5.xlsx") # saving intermediate data

In [34]:
# Model Evaluation with ATC Level 5 Codes Only

X, Y = model_input(ATC_only_5)
LR_ATC_5 = log_reg_boot100(X, Y)
for k , j in Y.items():
    print('This is the LR mean accuracy on x100 bootstrap for '+str(k), LR_ATC_5[k].mean())

This is the LR mean accuracy on x100 bootstrap for dizziness 0.5449491525423729
This is the LR mean accuracy on x100 bootstrap for nausea 0.5535329341317365
This is the LR mean accuracy on x100 bootstrap for headache 0.572246835443038
This is the LR mean accuracy on x100 bootstrap for rash 0.5695364238410595
This is the LR mean accuracy on x100 bootstrap for vomiting 0.5266776315789473
This is the LR mean accuracy on x100 bootstrap for asthenia 0.5579026217228464
This is the LR mean accuracy on x100 bootstrap for diarrhoea 0.5413043478260868
This is the LR mean accuracy on x100 bootstrap for pruritus 0.5851119402985073
This is the LR mean accuracy on x100 bootstrap for hypersensitivity 0.58032
This is the LR mean accuracy on x100 bootstrap for abdominal pain 0.5947234042553191
This is the LR mean accuracy on x100 bootstrap for urticaria 0.5830263157894737
This is the LR mean accuracy on x100 bootstrap for body temperature increased 0.5675892857142857
This is the LR mean accuracy on x10

In [35]:
from statsmodels.stats.anova import AnovaRM
import pandas as pd

lr_ANOVA = pd.DataFrame()

for j, i in enumerate(side_effect_count['Side Effect']):
    ANOVA = pd.DataFrame()
    ANOVA = pd.concat((ANOVA, pd.DataFrame({'model accuracy': LR_ATC[i], 'condition': 'LR: ATC Level 2'})))
    ANOVA = pd.concat((ANOVA, pd.DataFrame({'model accuracy': LR_ATC_3[i], 'condition': 'LR: ATC Level 3'})))
    ANOVA = pd.concat((ANOVA, pd.DataFrame({'model accuracy': LR_ATC_4[i], 'condition': 'LR: ATC Level 4'})))
    ANOVA = pd.concat((ANOVA, pd.DataFrame({'model accuracy': LR_ATC_5[i], 'condition': 'LR: ATC Level 5'})))
    
    ANOVA.reset_index(inplace=True)
    results = AnovaRM(data=ANOVA, depvar='model accuracy', subject='index', within=['condition']).fit()

    lr_ANOVA = pd.concat((lr_ANOVA, pd.DataFrame({
        'Side Effect': i, 
        'LR: ATC Level 2': LR_ATC[i].mean(),
        'LR: ATC Level 3': LR_ATC_3[i].mean(),
        'LR: ATC Level 4': LR_ATC_4[i].mean(),
        'LR: ATC Level 5': LR_ATC_5[i].mean(),
        'F-Value': results.anova_table['F Value'][0], 
        'P-value': results.anova_table['Pr > F'][0]
    }, index=[j])))

lr_ANOVA.to_excel("LR_ANOVA_ATC_Level234.xlsx")
lr_ANOVA


/var/folders/0x/qp6y3wx113b2t9y010wfvr5h0000gn/T/ipykernel_93295/3016437219.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'F-Value': results.anova_table['F Value'][0],
/var/folders/0x/qp6y3wx113b2t9y010wfvr5h0000gn/T/ipykernel_93295/3016437219.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'P-value': results.anova_table['Pr > F'][0]
/var/folders/0x/qp6y3wx113b2t9y010wfvr5h0000gn/T/ipykernel_93295/3016437219.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a 

,Side Effect,LR: ATC Level 2,LR: ATC Level 3,LR: ATC Level 4,LR: ATC Level 5,F-Value,P-value
0,dizziness,0.668203,0.664983,0.690068,0.544949,1162.697717,9.588567e-164
1,nausea,0.675210,0.680749,0.678563,0.553533,1223.332096,9.023775e-167
2,headache,0.677880,0.679937,0.683070,0.572247,845.969251,4.148129e-145
3,rash,0.677914,0.675662,0.683477,0.569536,1004.848805,3.971030e-155
4,vomiting,0.692401,0.688717,0.672796,0.526678,1668.967189,1.693920e-185
5,asthenia,0.716816,0.719700,0.711273,0.557903,1457.408848,2.797037e-177
6,diarrhoea,0.695507,0.695616,0.701413,0.541304,1679.732977,6.877678e-186
7,pruritus,0.712127,0.712201,0.704739,0.585112,768.132493,1.443479e-139
8,hypersensitivity,0.684240,0.693440,0.663080,0.580320,515.993141,2.023805e-117
9,abdominal pain,0.699872,0.700936,0.689106,0.594723,509.023552,1.098374e-116


# ATC and Drug Targets Comparison

In [ ]:
# adding drug targets as features
tar_ATC = pd.read_excel("./intermediate_data/tar_atc.xlsx")

print(tar_ATC.head())

In [ ]:
# Model Evaluation with Drugbank Targets and ATC Codes

X, Y = model_input(tar_ATC)
LR_Tar_ATC = log_reg_boot100(X, Y)
for k , j in Y.items():
    print('This is the LR mean accuracy on x100 bootstrap for '+str(k), LR_Tar_ATC[k].mean())